# OpenMP Mandelbrot

Flang compiles the loop with OpenMP, `libomp` runs it on native threads, and the kernel publishes the generated BMP directly as Jupyter `image/bmp` display data. There are no temporary image files and no user-side Base64 code.

In [ ]:
module bmp_display
  use xflang_display, only: display_bytes
  implicit none
contains
  subroutine display_image(w, h, rgba)
    integer, intent(in) :: w, h, rgba(4,w,h)
    integer :: rs, pad, fs, r, c, idx
    character(len=:), allocatable :: bmp
    rs = w*3; pad = mod(4-mod(rs,4),4); fs = 54+(rs+pad)*h
    allocate(character(len=fs) :: bmp); bmp = repeat(char(0),fs)
    bmp(1:2) = 'BM'
    call le32(bmp,3,fs); call le32(bmp,11,54)
    call le32(bmp,15,40); call le32(bmp,19,w); call le32(bmp,23,h)
    call le16(bmp,27,1); call le16(bmp,29,24)
    idx = 55
    do r = h, 1, -1
      do c = 1, w
        bmp(idx:idx)     = char(min(255,max(0,rgba(3,c,r))))
        bmp(idx+1:idx+1) = char(min(255,max(0,rgba(2,c,r))))
        bmp(idx+2:idx+2) = char(min(255,max(0,rgba(1,c,r))))
        idx = idx + 3
      end do
      idx = idx + pad
    end do
    call display_bytes('image/bmp', bmp)
  end subroutine
  subroutine le32(s,o,v)
    character(len=*),intent(inout)::s
    integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
    s(o+2:o+2)=char(iand(ishft(v,-16),255)); s(o+3:o+3)=char(iand(ishft(v,-24),255))
  end subroutine
  subroutine le16(s,o,v)
    character(len=*),intent(inout)::s
    integer,intent(in)::o,v
    s(o:o)=char(iand(v,255)); s(o+1:o+1)=char(iand(ishft(v,-8),255))
  end subroutine
end module bmp_display

In [ ]:
integer, parameter :: Nx = 600, Ny = 450, n_max = 255, dp = kind(0.d0)
real(dp), parameter :: xcenter = -0.5_dp, ycenter = 0.0_dp
real(dp), parameter :: width = 4.0_dp, height = 3.0_dp
real(dp), parameter :: dx_di = width/Nx, dy_dj = -height/Ny
real(dp), parameter :: x_offset = xcenter - (Nx+1)*dx_di/2
real(dp), parameter :: y_offset = ycenter - (Ny+1)*dy_dj/2

In [ ]:
integer :: image(Nx, Ny), image_color(4, Nx, Ny), palette(3, 4)
integer :: i, j, n, idx

In [ ]:
block
  integer :: pixel_i, pixel_j, iteration
  real(dp) :: zx, zy, cx, cy, zx2, zy2
  !$omp parallel do schedule(dynamic) private(pixel_i,iteration,zx,zy,cx,cy,zx2,zy2)
  do pixel_j = 1, Ny
    cy = y_offset + dy_dj * pixel_j
    do pixel_i = 1, Nx
      cx = x_offset + dx_di * pixel_i
      zx = 0; zy = 0; iteration = 0
      do
        zx2 = zx**2; zy2 = zy**2
        if (zx2 + zy2 > 4 .or. iteration == n_max) then
          image(pixel_i,pixel_j) = 255 - iteration
          exit
        end if
        zy = cy + 2*zx*zy
        zx = cx + zx2 - zy2
        iteration = iteration + 1
      end do
    end do
  end do
  !$omp end parallel do
end block
print *, 'OpenMP computation done.'

In [ ]:
palette(1,1) =   0; palette(2,1) = 135; palette(3,1) =  68
palette(1,2) =   0; palette(2,2) =  87; palette(3,2) = 231
palette(1,3) = 214; palette(2,3) =  45; palette(3,3) =  32
palette(1,4) = 255; palette(2,4) = 167; palette(3,4) =   0
do j = 1, Ny
  do i = 1, Nx
    idx = mod(image(i,j), 4) + 1
    image_color(1,i,j) = palette(1,idx)
    image_color(2,i,j) = palette(2,idx)
    image_color(3,i,j) = palette(3,idx)
    image_color(4,i,j) = 255
  end do
end do

In [ ]:
use bmp_display
call display_image(Nx, Ny, image_color)